In [1]:
# =========================
# 0) 必须最先设置 cache 到数据盘（在 import datasets 之前）
# =========================
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_CACHE"] = "/root/autodl-tmp/hf_datasets_cache"
os.environ["TRANSFORMERS_CACHE"] = "/root/autodl-tmp/hf_models_cache"
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from glob import glob

from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, ASTForAudioClassification

DATA_DIR = "./ASV_Spoof_2019_LA_SNR_50MB"
MODEL_DIR = "./mattyb95"

AUDIO_COL = "wav"
PARQUET_KEY_COL = "__key__"
JSONL_KEY_COL = "member"
JSONL_LABEL_COL = "key"

assert torch.cuda.is_available(), "CUDA 不可用"
device = torch.device("cuda")
print("CUDA OK:", torch.cuda.get_device_name(0))

# 1) parquet files
base = os.path.join(DATA_DIR, "default")
train_files = sorted(glob(os.path.join(base, "partial-train", "*.parquet")))
assert len(train_files) > 0, "没找到 train parquet"

# 2) load_dataset 会写 cache，但现在写到 /root/autodl-tmp 了
ds = load_dataset("parquet", data_files={"train": train_files})
train_ds = ds["train"].cast_column(AUDIO_COL, Audio(sampling_rate=16000))
print("train_ds loaded:", len(train_ds))

# 3) labels from jsonl
jsonl_path = os.path.join(DATA_DIR, "index", "train.jsonl")
assert os.path.isfile(jsonl_path), "找不到 train.jsonl"

member2label = {}
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        m = obj.get(JSONL_KEY_COL)
        k = obj.get(JSONL_LABEL_COL)
        if m is None or k is None:
            continue
        label = int(k) if isinstance(k, int) else (1 if str(k).lower() == "bonafide" else 0)
        member2label[str(m)] = int(label)
print("labels loaded:", len(member2label))

# 4) extractor + model (local only)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR, local_files_only=True)

model = ASTForAudioClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label={0: "spoof", 1: "bonafide"},
    label2id={"spoof": 0, "bonafide": 1},
    ignore_mismatched_sizes=True,
    local_files_only=True,
).to(device)
model.train()
print("model device:", next(model.parameters()).device)

# 5) collator
class ASTDataCollator:
    def __init__(self, fe, m2l, sr=16000, max_sec=6.0):
        self.fe = fe
        self.m2l = m2l
        self.max_len = int(sr * max_sec)

    def __call__(self, features):
        audios, labels = [], []
        for ex in features:
            kk = str(ex[PARQUET_KEY_COL])
            if kk not in self.m2l:
                raise ValueError(f"label missing for member={kk}")
            labels.append(self.m2l[kk])

            x = ex[AUDIO_COL]["array"]
            x = np.asarray(x, dtype=np.float32)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            if len(x) >= self.max_len:
                x = x[: self.max_len]
            else:
                x = np.pad(x, (0, self.max_len - len(x)))
            audios.append(x)

        inputs = self.fe(audios, sampling_rate=16000, return_tensors="pt")
        inputs["labels"] = torch.tensor(labels, dtype=torch.long)
        return inputs

train_collator = ASTDataCollator(feature_extractor, member2label)

# 6) manual train loop (10 steps)
dl = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0, collate_fn=train_collator)
optim = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("\nSTART MANUAL TRAIN LOOP")
for step, batch in enumerate(dl):
    if step >= 10:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    loss = model(**batch).loss
    loss.backward()
    optim.step()
    optim.zero_grad()
    print(f"step {step} | loss={loss.item():.4f}")
print("DONE")


FileNotFoundError: No parquet matched for train: ../asv_orig/train/*.parquet

In [2]:
!python train_wav2vec_base.py

/root/miniconda3/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniconda3/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/miniconda3/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we sus

In [3]:
!python train_hubert.py

/root/miniconda3/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniconda3/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/miniconda3/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we sus

In [1]:
!python train_wav2vec_base.py

/root/miniconda3/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniconda3/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/miniconda3/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we sus

In [2]:
!python train_ast.py

/root/miniconda3/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/root/miniconda3/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniconda3/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/miniconda3/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. 

In [9]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from datasets import Audio

# ========= 1. device =========
device = torch.device("cuda")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

# ========= 2. 本地模型路径 =========
MODEL_DIR = "./ast-finetuned-audioset-10-10-0.4593"

# ========= 3. feature extractor =========
feature_extractor = AutoFeatureExtractor.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

# ========= 4. model =========
id2label = {0: "spoof", 1: "bonafide"}
label2id = {"spoof": 0, "bonafide": 1}

model = ASTForAudioClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    local_files_only=True,
).to(device)

model.train()
print("Model device:", next(model.parameters()).device)

# ========= 5. 确保 train_ds 已 cast =========
train_ds = train_ds.cast_column("wav", Audio(sampling_rate=16000))

# ========= 6. DataLoader =========
dl = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    collate_fn=train_collator,  # 你之前定义的 ASTDataCollator
)

optim = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("START MANUAL TRAIN LOOP")

for step, batch in enumerate(dl):
    if step >= 10:
        break

    batch = {k: v.to(device) for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss

    loss.backward()
    optim.step()
    optim.zero_grad()

    print(f"step {step} | loss = {loss.item():.4f}")

print("DONE")


CUDA: True NVIDIA GeForce RTX 4090


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at ./ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model device: cuda:0


NameError: name 'train_ds' is not defined

In [15]:
!du -sh ~/.cache/huggingface/datasets 2>/dev/null
!rm -rf ~/.cache/huggingface/datasets


0	/root/.cache/huggingface/datasets


In [14]:
# ===== 必须在 import datasets 之前 =====
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import json
import numpy as np
import torch
from glob import glob
from torch.utils.data import DataLoader

from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, ASTForAudioClassification

DATA_DIR = "./ASV_Spoof_2019_LA_SNR_50MB"
MODEL_DIR = "./ast-finetuned-audioset-10-10-0.4593"

AUDIO_COL = "wav"
PARQUET_KEY_COL = "__key__"
JSONL_KEY_COL = "member"
JSONL_LABEL_COL = "key"

assert torch.cuda.is_available()
device = torch.device("cuda")
print("CUDA OK:", torch.cuda.get_device_name(0))

# 1) parquet file list
base = os.path.join(DATA_DIR, "default")
train_files = sorted(glob(os.path.join(base, "partial-train", "*.parquet")))
assert train_files, "没找到 train parquet"

# ✅ 2) streaming read (不会生成新的 arrow cache)
ds_stream = load_dataset(
    "parquet",
    data_files={"train": train_files},
    streaming=True,
)
train_stream = ds_stream["train"]

# buffer shuffle（流式数据集不能真正全局 shuffle）
train_stream = train_stream.shuffle(buffer_size=20000, seed=42)

# 3) jsonl -> member2label
jsonl_path = os.path.join(DATA_DIR, "index", "train.jsonl")
assert os.path.isfile(jsonl_path), "找不到 train.jsonl"

member2label = {}
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        m = obj.get(JSONL_KEY_COL)
        k = obj.get(JSONL_LABEL_COL)
        if m is None or k is None:
            continue
        label = int(k) if isinstance(k, int) else (1 if str(k).lower() == "bonafide" else 0)
        member2label[str(m)] = int(label)
print("labels loaded:", len(member2label))

# 4) model & extractor (local)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR, local_files_only=True)

model = ASTForAudioClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label={0: "spoof", 1: "bonafide"},
    label2id={"spoof": 0, "bonafide": 1},
    ignore_mismatched_sizes=True,
    local_files_only=True,
).to(device)
model.train()

# 5) collator：流式数据里 wav 可能是 bytes，需要自己解码
#   最稳：用 soundfile 读 bytes（你环境一般有）
import io
import soundfile as sf

class StreamCollator:
    def __init__(self, fe, m2l, sr=16000, max_sec=6.0):
        self.fe = fe
        self.m2l = m2l
        self.sr = sr
        self.max_len = int(sr * max_sec)

    def __call__(self, batch):
        audios, labels = [], []
        for ex in batch:
            kk = str(ex[PARQUET_KEY_COL]) + ".wav"
            if kk not in self.m2l:
                raise ValueError(f"label missing for member={kk}")
            labels.append(self.m2l[kk])

            w = ex[AUDIO_COL]

            # parquet 里常见两种：dict {'bytes':..., 'path':...} 或直接 bytes
            if isinstance(w, dict) and "bytes" in w and w["bytes"] is not None:
                data = w["bytes"]
                x, sr0 = sf.read(io.BytesIO(data), dtype="float32")
            elif isinstance(w, (bytes, bytearray)):
                x, sr0 = sf.read(io.BytesIO(w), dtype="float32")
            else:
                # 如果是已经解码好的 array
                x = w["array"] if isinstance(w, dict) and "array" in w else w
                sr0 = self.sr

            x = np.asarray(x, dtype=np.float32)
            if x.ndim > 1:
                x = x.mean(axis=-1)

            # 简单重采样：如果 sr0 != 16000，先粗暴处理（建议你数据本来就是 16k）
            if sr0 != self.sr:
                # 只做最简单的比例抽样（够用来验证训练是否跑起来）
                idx = np.linspace(0, len(x)-1, int(len(x)*self.sr/sr0)).astype(np.int64)
                x = x[idx]

            if len(x) >= self.max_len:
                x = x[:self.max_len]
            else:
                x = np.pad(x, (0, self.max_len - len(x)))

            audios.append(x)

        inputs = self.fe(audios, sampling_rate=self.sr, return_tensors="pt")
        inputs["labels"] = torch.tensor(labels, dtype=torch.long)
        return inputs

collator = StreamCollator(feature_extractor, member2label)

# 6) dataloader + manual train
dl = DataLoader(train_stream, batch_size=2, num_workers=0, collate_fn=collator)
optim = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("START 10 STEPS")
for step, batch in enumerate(dl):
    if step >= 10:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    loss = model(**batch).loss
    loss.backward()
    optim.step()
    optim.zero_grad()
    print("step", step, "loss", float(loss))
print("DONE")


CUDA OK: NVIDIA GeForce RTX 4090
labels loaded: 126900


Some weights of ASTForAudioClassification were not initialized from the model checkpoint at ./ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


START 10 STEPS
step 0 loss 0.9517346024513245
step 1 loss 0.8692223429679871
step 2 loss 0.11592144519090652
step 3 loss 0.07542797923088074
step 4 loss 0.004485085606575012
step 5 loss 0.02318951115012169
step 6 loss 0.009043055586516857
step 7 loss 4.159683704376221
step 8 loss 0.001284529105760157
step 9 loss 0.0005470791948027909
DONE
